# Import libraries and Load the Data

This notebook is a basic application of the XGBoost model with minimal feature engineering and tuning. 

In [ ]:
import pandas as pd

# Load datasets (already available in input/ directory)
train = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/train.csv', parse_dates=['date'])
test = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/test.csv', parse_dates=['date'])
oil = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/oil.csv', parse_dates=['date'])
holidays = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/holidays_events.csv', parse_dates=['date'])
transactions = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/transactions.csv', parse_dates=['date'])
stores = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/stores.csv')

# Quick check
print(train.head())
print("Train shape:", train.shape)

# Quick EDA


Data Overview

In [ ]:
# Check first rows and data types
print(train.head())
print(train.info())

# Summary of numerical columns
print(train.describe())

Missing Values

In [ ]:
# Count missing values per column
train.isnull().sum()

Sales Trend Over Time

In [ ]:
# Daily total sales
train.groupby('date')['sales'].sum().plot(figsize=(12,4), title='Total Sales Over Time')

Average Sales by Day of Week

In [ ]:
train.groupby(train['date'].dt.dayofweek)['sales'].mean().plot(
    kind='bar', title='Average Sales by Day of Week'
)

Sales by Month

In [ ]:
train.groupby(train['date'].dt.month)['sales'].mean().plot(
    kind='bar', title='Average Sales by Month'
)

Store-Level Analysis

In [ ]:
train.groupby('store_nbr')['sales'].mean().sort_values(ascending=False).plot(
    kind='bar', figsize=(12,4), title='Average Sales per Store'
)

Product Family Analysis

In [ ]:
train.groupby('family')['sales'].mean().sort_values(ascending=False).plot(
    kind='barh', figsize=(8,6), title='Average Sales per Product Family'
)

# Merging Extra Files
Goal: Enhance train/test sets with oil, holiday, transaction, and store data.

In [ ]:
# --- Load all datasets ---
train = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/train.csv', parse_dates=['date'])
test = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/test.csv', parse_dates=['date'])
oil = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/oil.csv', parse_dates=['date'])
holidays = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/holidays_events.csv', parse_dates=['date'])
transactions = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/transactions.csv', parse_dates=['date'])
stores = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/stores.csv')

# --- Merge Oil Prices ---
oil['dcoilwtico'] = oil['dcoilwtico'].ffill()
train = train.merge(oil, on='date', how='left')
test = test.merge(oil, on='date', how='left')

# --- Merge Holidays ---
holidays['is_holiday'] = 1
holidays = holidays[['date', 'is_holiday']].drop_duplicates(subset='date')
train = train.merge(holidays, on='date', how='left')
test = test.merge(holidays, on='date', how='left')
train['is_holiday'] = train['is_holiday'].fillna(0)
test['is_holiday'] = test['is_holiday'].fillna(0)

# --- Merge Transactions ---
train = train.merge(transactions, on=['date', 'store_nbr'], how='left')
test = test.merge(transactions, on=['date', 'store_nbr'], how='left')
train['transactions'] = train['transactions'].fillna(0)
test['transactions'] = test['transactions'].fillna(0)

# --- Merge Store Metadata ---
train = train.merge(stores, on='store_nbr', how='left')
test = test.merge(stores, on='store_nbr', how='left')

print("Train shape after merges:", train.shape)
print("Test shape after merges:", test.shape)


# Feature Engineering (Simple Baseline)

In [ ]:
#Convert categorical column (family) to numeric codes
train['family_encoded'] = train['family'].astype('category').cat.codes
test['family_encoded'] = test['family'].astype('category').cat.codes

# Encode object columns to numeric
for col in ['city', 'state', 'type']:
    if col in train.columns:
        train[col] = train[col].astype('category').cat.codes
        test[col] = test[col].astype('category').cat.codes


#Add date-based feature
train['day_of_week'] = train['date'].dt.dayofweek
train['month'] = train['date'].dt.month
train['year'] = train['date'].dt.year
train['day_of_month'] = train['date'].dt.day
train['week_of_year'] = train['date'].dt.isocalendar().week.astype(int)

test['day_of_week'] = test['date'].dt.dayofweek
test['month'] = test['date'].dt.month
test['year'] = test['date'].dt.year
test['day_of_month'] = test['date'].dt.day
test['week_of_year'] = test['date'].dt.isocalendar().week.astype(int)

#Weekend Flag
train['is_weekend'] = train['day_of_week'].isin([5, 6]).astype(int)
test['is_weekend'] = test['day_of_week'].isin([5, 6]).astype(int)

#Store type encoding
train['store_type_encoded'] = train['type'].astype('category').cat.codes
test['store_type_encoded'] = test['type'].astype('category').cat.codes

#Holiday Interaction
# Higher sales expected just before holidays
train['holiday_lag'] = train.groupby('store_nbr')['is_holiday'].shift(-1).fillna(0)
test['holiday_lag'] = test['is_holiday']  # no shift for test


# Baseline Model with XGBoost

In [ ]:
#Import XGBoost
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_log_error
import numpy as np

#Prepare Features and Target
# Target variable
y = train['sales']

# Features to use (add/remove as needed)
# Drop columns not used as features
drop_cols = ['id', 'date', 'sales', 'family']  # keep family_encoded

# Select all other columns as features
features = [col for col in train.columns if col not in drop_cols]

# Prepare training and test data
X = train[features]
X_test = test[features]

#Train/Test Split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, shuffle=False
)  # No shuffling due to time-series nature

#Train XGBoost Model
model = xgb.XGBRegressor(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
model.fit(X_train, y_train)

# Validation performance
val_pred = model.predict(X_val)
rmsle = np.sqrt(mean_squared_log_error(y_val, np.maximum(0, val_pred)))
print("Validation RMSLE:", rmsle)

#Generate Submission
# Predict on test set
test['sales'] = np.maximum(0, model.predict(X_test))

# Create submission
submission = test[['id', 'sales']]
submission.to_csv('submission.csv', index=False)